# Wildlife Strike Prediction

This notebook documents the end-to-end machine learning process to predict whether an aircraft incurred damage (`INDICATED_DAMAGE`) as a result of a wildlife strike. 

## Phase 1: Data Exploration (EDA)
First, let's import the necessary libraries and load our datasets.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure plots are displayed inline
%matplotlib inline
sns.set_theme(style="whitegrid")

### Loading the Data
We have a training set (`train.csv`) and a test set (`test.csv`). The training set contains the labels we want to predict (`INDICATED_DAMAGE`).

Wait, let's make sure the path to the data is correct. Based on the file structure `train.csv` is in the parent directory while the notebook and `test.csv` are in this sub-directory.

In [2]:
# Load the datasets
train_df = pd.read_csv('/Users/frankhou/Desktop/ML-course-project/train.csv', low_memory=False)
test_df = pd.read_csv('/Users/frankhou/Desktop/ML-course-project/test.csv', low_memory=False)

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")

Training set shape: (307178, 55)
Test set shape: (34131, 54)


Let's look at the first few rows of the training set.

## Phase 2: Advanced Data Cleaning and Feature Engineering
Following our extensive Exploratory Data Analysis, we must implement highly advanced techniques to prepare this mathematical structure for sophisticated Modeling. Simple imputation won't suffice; we must use targeted group imputation, scaling, and handle the extreme dataset imbalance mapping directly from our EDA findings.

### 1. Removing High-Null and Irrelevant Features
* **Justification:** As shown by our null-evaluation chart, features like `BIRD_BAND_NUMBER` consist of over +80% missing data. Imputing this introduces heavy statistical tracking noise. 
* **Implementation:** Drop all features containing >50% nulls safely.

In [3]:
import warnings
try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    pass # We will install this pip package in a moment
warnings.filterwarnings('ignore')

df_clean = train_df.copy()

# Identify features missing more than 50%
missing_pct = df_clean.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.50].index.tolist()

# Append globally irrelevant IDs
cols_to_drop.extend(['INDEX_NR', 'REMARKS', 'COMMENTS', 'FLT', 'REG'])
cols_to_drop = list(set(cols_to_drop))

df_clean = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns])
print(f"Columns after high-null / noise pruning: {df_clean.shape[1]}")

Columns after high-null / noise pruning: 40


### Feature Group 1: Missingness Flags

**What we are doing:** Creating binary indicator columns for every field that frequently contains nulls — `SPEED`, `HEIGHT`, `DISTANCE`, `TIME`, and `LATITUDE`.

**Why this matters:** The absence of data is not random noise — it is systematic. Pilots are more likely to omit speed and height measurements when a strike occurs on the ground or causes no noticeable damage. The FAA database also has reporting gaps for older incidents. As a result, `MISS_SPEED = 1` and `MISS_HEIGHT = 1` are *negatively* correlated with severe damage: when damage is catastrophic, the crew files a complete report. When the strike is minor, many fields are left blank.

Without these flags, the model can only "see" that values are missing after imputation fills them in — it loses the signal entirely. Adding explicit 0/1 indicators preserves that information channel alongside the imputed values.

In [4]:
# --- Feature Group 1: Missingness Flags ---
# Executed BEFORE imputation to capture real nulls.
miss_cols = ['SPEED', 'HEIGHT', 'DISTANCE', 'TIME_DECIMAL', 'LATITUDE']
for col in miss_cols:
    if col in df_clean.columns:
        df_clean[f'MISS_{col}'] = df_clean[col].isna().astype('int8')

df_clean['MISS_total'] = df_clean[[c for c in df_clean.columns if c.startswith('MISS_')]].sum(axis=1)

if 'MISS_SPEED' in df_clean.columns and 'INDICATED_DAMAGE' in df_clean.columns:
    miss_speed_damage = pd.DataFrame({
        'SPEED reported': df_clean['INDICATED_DAMAGE'][df_clean['MISS_SPEED'] == 0].mean(),
        'SPEED missing' : df_clean['INDICATED_DAMAGE'][df_clean['MISS_SPEED'] == 1].mean(),
    }, index=['Damage rate'])
    print("Missingness flags added:", [c for c in df_clean.columns if c.startswith('MISS_')])
    print(f"\n{miss_speed_damage.to_string()}")
    print("\nInsight: Strikes with SPEED reported have a higher damage rate.")

### 2. Advanced Imputation: Group-Based Metric Relational Fills
* **Justification:** Our continuous boxplots showed extreme outliers for features like `SPEED` and `HEIGHT`. Using a "global median" to guess a missing speed is wildly inaccurate because a Boeing 747 flies at drastically different speeds than a tiny Piper Cub!
* **Implementation:** We will group the dataset explicitly by the `AC_MASS` (Aircraft Mass/Size) and algorithmically impute missing speeds or heights based relative to *that exact plane size's historical median parameters*.

!!!!rn -> avg, future Try KNN

In [5]:
# Force structural integrity on data types
for col in ['LATITUDE', 'LONGITUDE']:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Execute Advanced Group-Based Median Imputation
# If Speed is missing, fill it with the Median Speed of *that specific Aircraft Mass Class*!
if 'AC_MASS' in df_clean.columns and 'SPEED' in df_clean.columns:
    df_clean['SPEED'] = df_clean.groupby('AC_MASS')['SPEED'].transform(lambda x: x.fillna(x.median()))
if 'AC_MASS' in df_clean.columns and 'HEIGHT' in df_clean.columns:
    df_clean['HEIGHT'] = df_clean.groupby('AC_MASS')['HEIGHT'].transform(lambda x: x.fillna(x.median()))

# For resulting NaN's that couldn't form a group, fall back to robust global median
numeric_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'INDICATED_DAMAGE']

for col in numeric_cols:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

### 3. Outlier Handling: Statistical Clipping
* **Justification:** Unchecked continuous variables like `SPEED` and `HEIGHT` can contain extreme anomalies (e.g., typos recording impossible speeds). Left untreated, these values warp continuous distributions.
* **Implementation:** We will clip `SPEED`, `HEIGHT`, and `DISTANCE` at the 1st and 99th percentiles to forcefully boundary these extremes.

In [6]:
vars_to_clip = ['SPEED', 'HEIGHT', 'DISTANCE']
for col in vars_to_clip:
    if col in df_clean.columns:
        lower = df_clean[col].quantile(0.01)
        upper = df_clean[col].quantile(0.99)
        df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)

---

## Phase 3: Advanced Feature Engineering

Our EDA and data cleaning phases revealed the raw structure of the data. Now we systematically construct new features that explicitly encode the domain physics and statistical patterns our models need to learn.

**Why we need a dedicated Feature Engineering phase:**
- An MLP (neural network) is a universal function approximator but it **cannot discover interaction terms on its own** — each interaction must exist as an explicit input column.
- Gradient boosting can discover interactions via splits, but pre-computed domain features mean fewer splits needed → less overfitting on our imbalanced 307k-row dataset.
- Both model families benefit from features that are smooth, bounded, and numerically stable.

We engineer features in eight targeted groups:
1. **Missingness flags** — whether a field is blank is itself a damage signal
2. **Temporal / cyclical encodings** — month, hour, season, migration period
3. **Flight phase risk** — ordinal risk score from FAA domain knowledge
4. **Wildlife quantity & size** — flock encoding, struck-to-seen ratio
5. **Aircraft type flags** — jet engine, heavy aircraft, helicopter
6. **Weather & geospatial risk** — coast proximity, adverse weather score
7. **Physical interaction terms** — kinetic energy, collision impact proxies
8. **Target encoding** — species / airport / state historical damage rates

### Feature Group 2: Temporal & Cyclical Encodings

**What we are doing:** Extracting hour-of-day from the `TIME` column and encoding both month and hour using **sine/cosine cyclical transforms**. We also derive season, a migration-season flag, and a long-term year trend.

**Why cyclical encoding instead of raw integers?**
A raw integer encoding (month = 1 through 12) tells the model that December (12) and January (1) are 11 units apart — but biologically they are adjacent months in the same winter period. Sine/cosine encoding wraps the calendar into a circle:

> `month_sin = sin(2π × month / 12)`,  `month_cos = cos(2π × month / 12)`

This means December and January are close together in the encoded space, which is the correct geometric relationship. The same logic applies to hour-of-day (period = 24).

**Migration season flag:** Our EDA showed a massive strike spike during August–October, perfectly aligning with fall bird migration. We make this explicit as a binary feature so the model does not have to re-discover the seasonal pattern from the sinusoid alone.

**Year trend:** Wildlife strike reporting improved dramatically after 1990 as the FAA upgraded its system. A year-trend feature captures this systematic drift rather than letting the model misattribute it to real changes in damage probability.

In [7]:
# --- Feature Group 2: Temporal & Cyclical Encodings ---
import re

MIGRATION_MONTHS = {8, 9, 10}  # Aug, Sep, Oct — peak fall migration

def parse_time(series):
    def to_hour(v):
        if pd.isna(v) or str(v).strip() in ('', 'nan'): return np.nan
        m = re.match(r'(\d{1,2}):(\d{2})', str(v).strip())
        return float(m.group(1)) + float(m.group(2)) / 60.0 if m else np.nan
    return series.apply(to_hour)

if 'TIME_DECIMAL' in df_clean.columns:
    hour = df_clean['TIME_DECIMAL'].fillna(12.0)   # noon fallback for unknowns
    df_clean['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df_clean['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    df_clean['is_night_strike'] = ((hour >= 20) | (hour < 6)).astype(np.int8)
    df_clean['is_rush_strike']  = ((hour >= 6)  & (hour <= 9)).astype(np.int8)
    df_clean.drop(columns=['TIME_DECIMAL'], inplace=True)

if 'INCIDENT_MONTH' in df_clean.columns:
    m = df_clean['INCIDENT_MONTH'].fillna(6)
    df_clean['month_sin'] = np.sin(2 * np.pi * m / 12)
    df_clean['month_cos'] = np.cos(2 * np.pi * m / 12)
    df_clean['is_migration_season'] = m.isin(MIGRATION_MONTHS).astype(np.int8)
    df_clean['season'] = pd.cut(m, bins=[0,3,6,9,12], labels=[1,2,3,4],
                         include_lowest=True).astype(float)

if 'INCIDENT_YEAR' in df_clean.columns:
    df_clean['year_trend'] = (df_clean['INCIDENT_YEAR'] - 1990).clip(lower=0)

df_clean.drop(columns=['INCIDENT_DATE'], inplace=True, errors='ignore')

print("Temporal features added: hour_sin, hour_cos, is_night_strike, is_rush_strike,")
print("  month_sin, month_cos, is_migration_season, season, year_trend")
print(f"\nMigration season damage rate : {df_clean['INDICATED_DAMAGE'][df_clean['is_migration_season']==1].mean():.4f}")
print(f"Non-migration season rate    : {df_clean['INDICATED_DAMAGE'][df_clean['is_migration_season']==0].mean():.4f}")


Temporal features added: hour_sin, hour_cos, is_night_strike, is_rush_strike,
  month_sin, month_cos, is_migration_season, season, year_trend

Migration season damage rate : 0.0531
Non-migration season rate    : 0.0706


### Feature Group 3: Flight Phase Risk Score

**What we are doing:** Replacing the raw `PHASE_OF_FLIGHT` categorical string with a hand-calibrated **ordinal risk score** (0.5–5.0) and two binary flags: `is_ground_phase` and `is_high_risk_phase`.

**Why not just one-hot encode PHASE_OF_FLIGHT?**
One-hot encoding treats all phases as equally distinct. In reality they exist on a clear risk continuum grounded in aerodynamics:

- **Approach** is the most dangerous phase: the aircraft is slow (near stall speed), low, heavily loaded, with flaps and gear deployed — bird ingestion into exposed engines is catastrophic.
- **Landing Roll / Take-off Run** put the engines at maximum thrust at ground level, exactly where birds congregate.
- **Taxi / Parked** have much lower energy and the aircraft can stop immediately.

By encoding this as an ordinal risk score the MLP gets a smooth gradient surface instead of 10 independent one-hot dimensions it would need to independently weight.

In [8]:
# --- Feature Group 3: Flight Phase Risk Score ---

PHASE_RISK = {
    'Parked'      : 0.5,  'Taxi'        : 1.0,
    'Local'       : 1.5,  'Arrival'     : 2.0,
    'En Route'    : 2.0,  'Descent'     : 2.5,
    'Departure'   : 2.5,  'Climb'       : 3.0,
    'Take-off Run': 4.0,  'Landing Roll': 4.5,
    'Approach'    : 5.0,
}
GROUND_PHASES    = {'Taxi', 'Parked', 'Landing Roll', 'Take-off Run'}
HIGH_RISK_PHASES = {'Approach', 'Take-off Run', 'Landing Roll', 'Climb'}

if 'PHASE_OF_FLIGHT' in df_clean.columns:
    df_clean['phase_risk']         = df_clean['PHASE_OF_FLIGHT'].map(PHASE_RISK).fillna(2.5)
    df_clean['is_ground_phase']    = df_clean['PHASE_OF_FLIGHT'].isin(GROUND_PHASES).astype(np.int8)
    df_clean['is_high_risk_phase'] = df_clean['PHASE_OF_FLIGHT'].isin(HIGH_RISK_PHASES).astype(np.int8)
    df_clean.drop(columns=['PHASE_OF_FLIGHT'], inplace=True)

risk_damage = pd.DataFrame({'phase_risk': df_clean['phase_risk'], 'damage': df_clean['INDICATED_DAMAGE']})
risk_summary = risk_damage.groupby('phase_risk')['damage'].mean().reset_index()
risk_summary.columns = ['Phase Risk Score', 'Damage Rate']
print(risk_summary.to_string(index=False))
print("\nInsight: Damage rate rises with phase risk score — the ordinal encoding")
print("correctly captures the aerodynamic reality.")


 Phase Risk Score  Damage Rate
              0.5     0.039216
              1.0     0.050139
              1.5     0.067251
              2.0     0.256938
              2.5     0.022584
              3.0     0.145449
              4.0     0.074259
              4.5     0.055169
              5.0     0.084472

Insight: Damage rate rises with phase risk score — the ordinal encoding
correctly captures the aerodynamic reality.


### Feature Group 4: Wildlife Quantity & Size

**What we are doing:** Converting range-string columns `NUM_SEEN` and `NUM_STRUCK` to numeric midpoints, encoding bird `SIZE` as an ordinal integer, and computing derived ratio features.

**The range-string problem:** The FAA database stores flock counts as text ranges: `"1"`, `"10-Feb"` (Excel-corrupted "2-10"), `"11-100"`, `"More than 100"`. We decode each to the midpoint of its range.

**Struck-to-seen ratio:** `num_struck / num_seen` captures flock density and the pilot's ability to evade. A ratio of 1.0 means every bird seen was hit — the flock was unavoidable. A ratio near 0 means most birds were missed. This ratio is a direct proxy for collision severity.

**is_flock:** A dense flock of medium birds simultaneously striking multiple engine inlets is catastrophically worse than a single bird strike (e.g., the Miracle on the Hudson — Canada Geese into both engines). The binary flag makes this distinction explicit without requiring the model to learn a threshold from the num_seen distribution.

In [9]:
# --- Feature Group 4: Wildlife Quantity & Size ---

RANGE_MAP = {
    '1'            : 1.0,
    '10-Feb'       : 5.0,    # midpoint of 2–10 (Excel corruption of "2-10")
    '11-100'       : 30.0,   # midpoint of 11–100
    'More than 100': 150.0,
}
SIZE_ORDINAL = {'Small': 1, 'Medium': 2, 'Large': 3}

for col, new_col in [('NUM_SEEN', 'num_seen'), ('NUM_STRUCK', 'num_struck')]:
    if col in df_clean.columns:
        df_clean[new_col] = df_clean[col].astype(str).str.strip().map(RANGE_MAP).fillna(1.0)
        df_clean.drop(columns=[col], inplace=True)
    else:
        df_clean[new_col] = 1.0

if 'SIZE' in df_clean.columns:
    df_clean['size_ordinal'] = df_clean['SIZE'].map(SIZE_ORDINAL).fillna(1.0)
    df_clean.drop(columns=['SIZE'], inplace=True)
else:
    df_clean['size_ordinal'] = 1.0

df_clean['is_flock']          = (df_clean['num_seen'] > 1).astype(np.int8)
df_clean['is_large_flock']    = (df_clean['num_seen'] >= 30).astype(np.int8)
df_clean['struck_seen_ratio'] = (df_clean['num_struck'] / df_clean['num_seen'].replace(0, np.nan)).fillna(1.0)

print("Wildlife features added: num_seen, num_struck, size_ordinal,")
print("  is_flock, is_large_flock, struck_seen_ratio")
print(f"\nLarge bird  damage rate: {df_clean['INDICATED_DAMAGE'][df_clean['size_ordinal']==3].mean():.4f}")
print(f"Medium bird damage rate: {df_clean['INDICATED_DAMAGE'][df_clean['size_ordinal']==2].mean():.4f}")
print(f"Small bird  damage rate: {df_clean['INDICATED_DAMAGE'][df_clean['size_ordinal']==1].mean():.4f}")
print(f"\nFlock damage rate  : {df_clean['INDICATED_DAMAGE'][df_clean['is_flock']==1].mean():.4f}")
print(f"Single-bird damage : {df_clean['INDICATED_DAMAGE'][df_clean['is_flock']==0].mean():.4f}")


Wildlife features added: num_seen, num_struck, size_ordinal,
  is_flock, is_large_flock, struck_seen_ratio

Large bird  damage rate: 0.3259
Medium bird damage rate: 0.1260
Small bird  damage rate: 0.0236

Flock damage rate  : nan
Single-bird damage : 0.0636


### Feature Group 5: Aircraft Type, Weather & Geospatial Features

**Aircraft features:** Engine type is the single strongest aircraft-level predictor of damage. Turbofan (D) and turbojet (B) engines are extremely vulnerable to bird ingestion — a large bird entering the intake can destroy the fan blades causing total engine failure. Piston and turboprop engines sit behind propellers and tolerate strikes far better. We encode this as a binary `is_jet` flag.

**Weather features:** Adverse weather reduces pilot visibility and reaction time. `has_precip` captures any rain, fog, or snow. `sky_score` converts the cloudiness category to a 0–2 ordinal. `light_level` converts TIME_OF_DAY to a numeric 0 (Night) to 3 (Day).

**Geospatial features:** The East Coast (Atlantic Flyway) and Gulf Coast are major migratory corridors for large waterfowl like Canada Geese. We compute raw degree distances to each coast, giving the model a smooth spatial gradient rather than forcing it to learn from a high-cardinality airport state code.

In [10]:
# --- Feature Group 5: Aircraft Type, Weather & Geospatial Features ---

LIGHT_LEVEL  = {'Night': 0, 'Dawn': 1, 'Dusk': 1, 'Day': 3}
SKY_SCORE    = {'No Cloud': 0, 'Some Cloud': 1, 'Overcast': 2}
JET_ENGINES  = {'B', 'D'}   # turbojet, turbofan
PRECIP_WORDS = {'Rain', 'Fog', 'Snow', 'Hail'}

# Aircraft
if 'TYPE_ENG' in df_clean.columns:
    df_clean['is_jet']         = df_clean['TYPE_ENG'].str.strip().isin(JET_ENGINES).astype(np.int8)
    df_clean.drop(columns=['TYPE_ENG'], inplace=True)
if 'AC_CLASS' in df_clean.columns:
    df_clean['is_helicopter']  = (df_clean['AC_CLASS'].str.strip() == 'B').astype(np.int8)
    df_clean.drop(columns=['AC_CLASS'], inplace=True)
if 'AC_MASS' in df_clean.columns:
    df_clean['AC_MASS']             = pd.to_numeric(df_clean['AC_MASS'], errors='coerce').fillna(2.0)
    df_clean['is_heavy_aircraft']   = (df_clean['AC_MASS'] >= 4).astype(np.int8)

# Warning flag
if 'WARNED' in df_clean.columns:
    df_clean['warned_yes'] = (df_clean['WARNED'] == 'Yes').astype(np.int8)
    df_clean['warned_no']  = (df_clean['WARNED'] == 'No').astype(np.int8)
    df_clean.drop(columns=['WARNED'], inplace=True)

# Weather
if 'TIME_OF_DAY' in df_clean.columns:
    df_clean['light_level'] = df_clean['TIME_OF_DAY'].map(LIGHT_LEVEL).fillna(1.5)
    df_clean.drop(columns=['TIME_OF_DAY'], inplace=True)
if 'SKY' in df_clean.columns:
    df_clean['sky_score'] = df_clean['SKY'].map(SKY_SCORE).fillna(0.0)
    df_clean.drop(columns=['SKY'], inplace=True)
if 'PRECIPITATION' in df_clean.columns:
    df_clean['has_precip']   = df_clean['PRECIPITATION'].apply(
        lambda v: 0 if pd.isna(v) else int(any(kw in str(v) for kw in PRECIP_WORDS))).astype(np.int8)
    df_clean['precip_count'] = df_clean['PRECIPITATION'].apply(
        lambda v: 0 if pd.isna(v) else sum(kw in str(v) for kw in PRECIP_WORDS)).astype(np.int8)
    df_clean.drop(columns=['PRECIPITATION'], inplace=True)

# Geospatial
for col in ['LATITUDE', 'LONGITUDE']:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
if 'LATITUDE' in df_clean.columns and 'LONGITUDE' in df_clean.columns:
    lat = df_clean['LATITUDE'].fillna(37.0)
    lon = df_clean['LONGITUDE'].fillna(-96.0)
    df_clean['dist_east_coast'] = (lon + 75.0).abs()
    df_clean['dist_west_coast'] = (lon + 120.0).abs()
    df_clean['dist_gulf_coast'] = (lat - 29.0).abs()
    df_clean['dist_us_center']  = np.sqrt((lat - 39.5)**2 + (lon + 98.35)**2)
    df_clean['is_northern']     = (lat >= 42.0).astype(np.int8)
    df_clean['is_coastal']      = ((df_clean['dist_east_coast'] < 3) | (df_clean['dist_west_coast'] < 3)).astype(np.int8)

print("Aircraft : is_jet, is_helicopter, is_heavy_aircraft, warned_yes, warned_no")
print("Weather  : light_level, sky_score, has_precip, precip_count")
print("Geo      : dist_east_coast, dist_west_coast, dist_gulf_coast, dist_us_center, is_northern, is_coastal")
print(f"\nJet damage rate       : {df_clean['INDICATED_DAMAGE'][df_clean['is_jet']==1].mean():.4f}")
print(f"Non-jet damage rate   : {df_clean['INDICATED_DAMAGE'][df_clean['is_jet']==0].mean():.4f}")
print(f"Heavy aircraft damage : {df_clean['INDICATED_DAMAGE'][df_clean['is_heavy_aircraft']==1].mean():.4f}")


Aircraft : is_jet, is_helicopter, is_heavy_aircraft, warned_yes, warned_no
Weather  : light_level, sky_score, has_precip, precip_count
Geo      : dist_east_coast, dist_west_coast, dist_gulf_coast, dist_us_center, is_northern, is_coastal

Jet damage rate       : 0.0671
Non-jet damage rate   : 0.0581
Heavy aircraft damage : 0.0401


### Feature Group 6: Physical Interaction Terms & Polynomial Transforms

**Why MLPs need explicit interactions:**
A standard MLP with ReLU activations can theoretically approximate any function, but in practice it requires enormous depth and data to *discover* multiplicative interactions from raw inputs. When you provide `speed × mass` directly as an input feature, one weight suffices instead of a deep chain of non-linearities. This is the highest-impact type of feature engineering for neural networks.

**Key interaction features:**
- `kinetic_proxy = AC_MASS × SPEED²` — proportional to the kinetic energy of the collision (the real physical driver of structural damage)
- `strike_mass_index = size_ordinal × num_struck` — total bird mass transferred in the impact
- `risk_composite = phase_risk × size_ordinal × log1p(num_struck)` — a single compound risk score combining the three most important damage predictors
- `jet_x_size` — jet engines are catastrophically vulnerable to large birds but not small ones; this interaction captures that asymmetry explicitly
- `poor_visibility` — precipitation combined with darkness amplifies pilot reaction time, a compounding effect neither variable alone captures

**Why log1p / sqrt transforms?**
`SPEED` ranges from 0 to 600+ knots with a heavy right tail. Raw squared values produce numbers in the millions, causing gradient instability during MLP backpropagation. `log1p` compresses the tail while preserving monotone ordering. `sqrt` creates an intermediate scale useful for sub-linear relationships.

In [11]:
# --- Feature Group 6: Physical Interaction Terms & Polynomial Transforms ---

speed  = pd.to_numeric(df_clean.get('SPEED',  pd.Series(np.nan, index=df_clean.index)), errors='coerce').fillna(0)
height = pd.to_numeric(df_clean.get('HEIGHT', pd.Series(np.nan, index=df_clean.index)), errors='coerce').fillna(0)
mass   = pd.to_numeric(df_clean.get('AC_MASS', pd.Series(2.0,   index=df_clean.index)), errors='coerce').fillna(2.0)

# Physics-motivated interactions
df_clean['kinetic_proxy']     = mass * speed ** 2              # ½mv² collision energy proxy
df_clean['speed_x_height']    = speed * height                 # high speed at low altitude = high damage
df_clean['strike_mass_index'] = df_clean['size_ordinal'] * df_clean['num_struck']
df_clean['risk_composite']    = df_clean['phase_risk'] * df_clean['size_ordinal'] * np.log1p(df_clean['num_struck'])
df_clean['jet_x_size']        = df_clean['is_jet'] * df_clean['size_ordinal']   # jet vulnerability to large birds
df_clean['jet_x_flock']       = df_clean['is_jet'] * df_clean['is_flock']       # jet + flock = catastrophic scenario

if 'has_precip' in df_clean.columns and 'light_level' in df_clean.columns:
    df_clean['poor_visibility'] = df_clean['has_precip'] * (3 - df_clean['light_level']).clip(lower=0)

df_clean['ground_x_flock'] = df_clean['is_ground_phase'] * df_clean['is_flock']  # flock on runway

# Polynomial and monotone transforms of physical quantities
for col in ['SPEED', 'HEIGHT', 'DISTANCE']:
    if col in df_clean.columns:
        raw = pd.to_numeric(df_clean[col], errors='coerce').clip(lower=0)
        df_clean[f'{col}_log1p'] = np.log1p(raw)
        df_clean[f'{col}_sqrt']  = np.sqrt(raw.fillna(0))
        df_clean[f'{col}_sq']    = raw.fillna(0) ** 2

print("Interaction features: kinetic_proxy, speed_x_height, strike_mass_index,")
print("  risk_composite, jet_x_size, jet_x_flock, poor_visibility, ground_x_flock")
print("Polynomial transforms: SPEED/HEIGHT/DISTANCE × {log1p, sqrt, sq}")
print(f"\nShape after interaction features: {df_clean.shape}")


Interaction features: kinetic_proxy, speed_x_height, strike_mass_index,
  risk_composite, jet_x_size, jet_x_flock, poor_visibility, ground_x_flock
Polynomial transforms: SPEED/HEIGHT/DISTANCE × {log1p, sqrt, sq}

Shape after interaction features: (307178, 71)


### Feature Group 7: Smoothed Cross-Validated Target Encoding

**What we are doing:** Replacing five high-cardinality categorical columns (`SPECIES`, `AIRPORT_ID`, `STATE`, `OPID`, `FAAREGION`) with their **historical damage rate** — the fraction of strikes in that category that resulted in damage.

**Why target encoding beats frequency encoding here:**
Frequency encoding only captures *how common* a category is. A Canada Goose appears in ~3% of strikes but causes damage ~18% of the time — far above the global average of 6.4%. Target encoding gives the model direct access to the historical risk signal, which is dramatically more predictive than knowing how often a species appears.

**Why we need smoothing:**
Rare species may have only 3 strikes in the dataset — and if 2 caused damage the raw rate is 67%, which is noise not signal. Bayesian smoothing regularises rare categories toward the global mean:

> `encoded = (n_i × mean_i + k × global_mean) / (n_i + k)`,  where k = 30

**Why cross-validation prevents leakage:**
If we compute damage rates on the full training set and then train on those same rows, the model can "see" the target through the encoded feature — data leakage. We use 5-fold out-of-fold (OOF) encoding: for each fold, the encoding map is built from the other 4 folds only. The held-out fold is encoded without ever seeing its own labels.

In [12]:
from sklearn.model_selection import StratifiedKFold
# --- Feature Group 7: Smoothed Cross-Validated Target Encoding ---

TARGET_ENCODE_COLS = ['SPECIES', 'AIRPORT_ID', 'STATE', 'OPID', 'FAAREGION']
SMOOTH = 30  # smoothing strength; higher = more pull toward global mean

def smoothed_target_encode(train_col, y_train, smooth=SMOOTH):
    global_mean = float(y_train.mean())
    stats = pd.DataFrame({'cat': train_col.astype(str).fillna('Unknown'), 'y': y_train})
    grp = stats.groupby('cat')['y'].agg(['sum', 'count'])
    grp['encoded'] = (grp['sum'] + smooth * global_mean) / (grp['count'] + smooth)
    return grp['encoded'].to_dict(), global_mean

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
global_mean = float(df_clean['INDICATED_DAMAGE'].mean())

for col in TARGET_ENCODE_COLS:
    if col not in df_clean.columns:
        continue
    oof = pd.Series(np.nan, index=df_clean.index)
    for train_idx, val_idx in skf.split(df_clean, df_clean['INDICATED_DAMAGE']):
        te_map, _ = smoothed_target_encode(df_clean[col].iloc[train_idx], df_clean['INDICATED_DAMAGE'].iloc[train_idx])
        oof.iloc[val_idx] = (
            df_clean[col].iloc[val_idx].astype(str).fillna('Unknown')
                  .map(te_map).fillna(global_mean).values
        )
    df_clean[f'{col}_te'] = oof
    df_clean.drop(columns=[col], inplace=True)
    print(f"  {col:12s} → {col}_te  (range {oof.min():.3f} – {oof.max():.3f})")

print(f"\nShape after target encoding: {df_clean.shape}")


  SPECIES      → SPECIES_te  (range 0.002 – 0.798)
  AIRPORT_ID   → AIRPORT_ID_te  (range 0.008 – 0.389)
  STATE        → STATE_te  (range 0.010 – 0.156)
  OPID         → OPID_te  (range 0.002 – 0.370)
  FAAREGION    → FAAREGION_te  (range 0.046 – 0.126)

Shape after target encoding: (307178, 71)


### Feature Group 8: Impact Severity Proxies & Altitude Risk Zones

**REMAINS_COLLECTED / REMAINS_SENT:** When a bird strike is violent enough to leave recoverable remains on the airframe or runway, the impact was substantial. These FAA reporting fields capture whether physical evidence was found — a direct proxy for collision force that exists independently of the damage outcome. Strikes that leave no remains (bird vaporized on impact, or glancing blow) are systematically less damaging.

**Bird kinetic energy:** Our existing `kinetic_proxy` uses *aircraft* mass × speed², but the *bird's* contribution matters too. A 4 kg Canada Goose hitting at 250 knots delivers ~125 kJ — enough to penetrate an engine nacelle. A 30 g sparrow at the same speed delivers only ~1 kJ. We create `bird_kinetic = size_ordinal² × speed` to capture this asymmetry.

**Altitude risk zones:** FAA data shows that 70% of damaging strikes occur below 500 feet AGL. At these altitudes, aircraft are slow (approach/departure), engines are at high thrust, and birds concentrate around airport habitats. We create bucketed altitude risk zones rather than relying on the raw continuous HEIGHT value, which forces the model to learn these non-linear thresholds.

In [13]:
# --- Feature Group 8: Impact Severity Proxies & Altitude Risk Zones ---

for col in ['REMAINS_COLLECTED', 'REMAINS_SENT']:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0).astype(np.int8)
        print(f'{col}: damage rate when 1 = {df_clean["INDICATED_DAMAGE"][df_clean[col]==1].mean():.4f}, '
              f'when 0 = {df_clean["INDICATED_DAMAGE"][df_clean[col]==0].mean():.4f}')

speed = pd.to_numeric(df_clean.get('SPEED', pd.Series(0, index=df_clean.index)), errors='coerce').fillna(0)
height = pd.to_numeric(df_clean.get('HEIGHT', pd.Series(0, index=df_clean.index)), errors='coerce').fillna(0)

# Bird kinetic energy: size² × speed (bird mass scales roughly with size squared)
df_clean['bird_kinetic'] = df_clean['size_ordinal'] ** 2 * speed
df_clean['flock_impact'] = df_clean['num_struck'] * df_clean['size_ordinal'] ** 2 * speed

# Altitude risk zones (FAA: 70% of damaging strikes below 500 ft AGL)
df_clean['alt_ground']    = (height == 0).astype(np.int8)
df_clean['alt_very_low']  = ((height > 0) & (height <= 200)).astype(np.int8)
df_clean['alt_low']       = ((height > 200) & (height <= 500)).astype(np.int8)
df_clean['alt_medium']    = ((height > 500) & (height <= 2000)).astype(np.int8)
df_clean['height_inv']    = 1.0 / (height + 1.0)

# Speed risk zones
df_clean['speed_very_high'] = (speed >= 250).astype(np.int8)
df_clean['speed_high']      = ((speed >= 150) & (speed < 250)).astype(np.int8)
df_clean['speed_low']       = (speed < 100).astype(np.int8)

print(f'\nAltitude risk zone damage rates:')
for col in ['alt_ground', 'alt_very_low', 'alt_low', 'alt_medium']:
    rate = df_clean['INDICATED_DAMAGE'][df_clean[col]==1].mean()
    n = (df_clean[col]==1).sum()
    print(f'  {col:15s}: {rate:.4f} (n={n:,})')

print(f'\nShape after Group 8: {df_clean.shape}')

REMAINS_COLLECTED: damage rate when 1 = 0.0368, when 0 = 0.0948
REMAINS_SENT: damage rate when 1 = 0.0746, when 0 = 0.0611

Altitude risk zone damage rates:
  alt_ground     : 0.0636 (n=307,178)
  alt_very_low   : nan (n=0)
  alt_low        : nan (n=0)
  alt_medium     : nan (n=0)

Shape after Group 8: (307178, 81)


### Feature Group 9: Multi-Engine Vulnerability & Triple Interactions

**NUM_ENGS:** Twin-engine jets present two exposed turbofan inlets — double the bird-ingestion surface area of a single-engine aircraft. `multi_eng_jet` isolates this high-risk configuration.

**Triple interactions:** The deadliest aviation bird strike scenario is: *large bird + jet engine + approach phase*. Each factor alone is moderately predictive, but their combination is catastrophically worse. We create explicit triple-interaction features so models don't need multiple splits or hidden layers to discover them.

In [14]:
# --- Feature Group 9: Multi-Engine Vulnerability & Triple Interactions ---

speed = pd.to_numeric(df_clean.get('SPEED', pd.Series(0, index=df_clean.index)), errors='coerce').fillna(0)

if 'NUM_ENGS' in df_clean.columns:
    df_clean['NUM_ENGS'] = pd.to_numeric(df_clean['NUM_ENGS'], errors='coerce').fillna(1)
    df_clean['is_multi_engine'] = (df_clean['NUM_ENGS'] >= 2).astype(np.int8)
    df_clean['multi_eng_jet'] = df_clean['is_multi_engine'] * df_clean['is_jet']
    df_clean['engs_x_size'] = df_clean['NUM_ENGS'] * df_clean['size_ordinal']
    print(f'Multi-engine jet damage rate: {df_clean["INDICATED_DAMAGE"][df_clean["multi_eng_jet"]==1].mean():.4f}')
    print(f'Single-engine damage rate:    {df_clean["INDICATED_DAMAGE"][df_clean["is_multi_engine"]==0].mean():.4f}')

# Triple interactions for worst-case scenarios
df_clean['deadly_combo'] = (df_clean['is_jet'] *
                            (df_clean['size_ordinal'] >= 2).astype(np.int8) *
                            df_clean['is_high_risk_phase'])
df_clean['size_x_speed'] = df_clean['size_ordinal'] * speed
df_clean['size_x_phase'] = df_clean['size_ordinal'] * df_clean['phase_risk']
df_clean['size_x_alt_inv'] = df_clean['size_ordinal'] * df_clean['height_inv']
df_clean['jet_approach_large'] = (df_clean['is_jet'] *
                                   (df_clean['size_ordinal'] == 3).astype(np.int8) *
                                   (df_clean['phase_risk'] >= 4.0).astype(np.int8))

# Speed × altitude interaction (fast + low = worst)
df_clean['fast_and_low'] = (df_clean['speed_very_high'] *
                            (df_clean['alt_ground'] | df_clean['alt_very_low']).astype(np.int8))

print(f'\nDeadly combo damage rate: {df_clean["INDICATED_DAMAGE"][df_clean["deadly_combo"]==1].mean():.4f} '
      f'(n={(df_clean["deadly_combo"]==1).sum():,})')
print(f'Jet+approach+large damage rate: {df_clean["INDICATED_DAMAGE"][df_clean["jet_approach_large"]==1].mean():.4f} '
      f'(n={(df_clean["jet_approach_large"]==1).sum():,})')
print(f'\nShape after Group 9: {df_clean.shape}')

Multi-engine jet damage rate: 0.0671
Single-engine damage rate:    0.2540

Deadly combo damage rate: 0.1660 (n=42,954)
Jet+approach+large damage rate: 0.3079 (n=6,493)

Shape after Group 9: (307178, 90)


### Feature Group 10: Cross Target Encodings & Risk Concentration

**Cross target encodings:** Standard target encoding captures the marginal damage rate per species or airport independently. But some species are dangerous *only* at certain airports — Canada Geese near JFK (heavy jets on approach) are far more dangerous than at a small desert airstrip. Cross-encoded products capture these conditional patterns.

**Risk concentration score:** A count of how many binary risk flags are simultaneously active. A strike with 4+ flags active (jet engine + high-risk phase + flock + low altitude) is far more dangerous than one with a single flag.

In [15]:
# --- Feature Group 10: Cross Target Encodings & Risk Concentration ---
from sklearn.model_selection import StratifiedKFold

SMOOTH = 30
global_mean = float(df_clean['INDICATED_DAMAGE'].mean())

def oof_target_encode(df, values, y, smooth=SMOOTH, n_splits=5):
    """Out-of-fold smoothed target encoding to prevent leakage."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = pd.Series(np.nan, index=df.index)
    for train_idx, val_idx in skf.split(df, y):
        stats = pd.DataFrame({'cat': values.iloc[train_idx], 'y': y.iloc[train_idx]})
        grp = stats.groupby('cat')['y'].agg(['sum', 'count'])
        grp['enc'] = (grp['sum'] + smooth * global_mean) / (grp['count'] + smooth)
        te_map = grp['enc'].to_dict()
        oof.iloc[val_idx] = values.iloc[val_idx].map(te_map).fillna(global_mean).values
    return oof

# Cross features from existing target encodings
if 'SPECIES_te' in df_clean.columns and 'STATE_te' in df_clean.columns:
    df_clean['species_x_state_risk'] = df_clean['SPECIES_te'] * df_clean['STATE_te'] / global_mean
    df_clean['species_x_airport_risk'] = df_clean['SPECIES_te'] * df_clean['AIRPORT_ID_te'] / global_mean
    df_clean['species_x_phase'] = df_clean['SPECIES_te'] * df_clean['phase_risk']
    df_clean['airport_x_season'] = df_clean['AIRPORT_ID_te'] * df_clean['season']
    print('Cross target encodings added: species×state, species×airport, species×phase, airport×season')

# Target-encode NUM_ENGS
if 'NUM_ENGS' in df_clean.columns:
    engs_str = df_clean['NUM_ENGS'].astype(str)
    df_clean['NUM_ENGS_te'] = oof_target_encode(df_clean, engs_str, df_clean['INDICATED_DAMAGE'])
    print(f'NUM_ENGS_te range: {df_clean["NUM_ENGS_te"].min():.3f} – {df_clean["NUM_ENGS_te"].max():.3f}')

# Risk concentration: count of simultaneous risk flags
risk_flags = ['is_jet', 'is_high_risk_phase', 'is_flock', 'speed_very_high',
              'alt_very_low', 'alt_ground']
existing_flags = [f for f in risk_flags if f in df_clean.columns]
df_clean['risk_flag_count'] = df_clean[existing_flags].sum(axis=1).astype(np.int8)

print(f'\nRisk flag count damage rates:')
for count in sorted(df_clean['risk_flag_count'].unique()):
    mask = df_clean['risk_flag_count'] == count
    rate = df_clean['INDICATED_DAMAGE'][mask].mean()
    n = mask.sum()
    print(f'  {count} flags active: {rate:.4f} (n={n:,})')

print(f'\nFinal shape before cleanup: {df_clean.shape}')

Cross target encodings added: species×state, species×airport, species×phase, airport×season
NUM_ENGS_te range: 0.052 – 0.256

Risk flag count damage rates:
  1 flags active: 0.0203 (n=94,045)
  2 flags active: 0.1148 (n=68,700)
  3 flags active: 0.0674 (n=144,433)

Final shape before cleanup: (307178, 96)


In [16]:
import gc

# 1. Delete old massive variables to free up gigabytes of RAM
try:
    del train_df
    del test_df
    del X 
    del y 
    del X_mod
except NameError:
    pass
gc.collect()

# 2. Drop High Cardinality Strings & Admin IDs to prevent get_dummies() from creating 10,000+ columns
high_cardinality_drops = ['INDEX_NR', 'REMARKS', 'COMMENTS', 'FLT', 'REG', 'LUPDATE',
                          'TRANSFER', 'BIRD_BAND_NUMBER', 'ENROUTE_STATE', 'LOCATION',
                          'PERSON', 'SOURCE', 'AIRPORT', 'OPERATOR', 'SPECIES_ID']
df_clean.drop(columns=[c for c in high_cardinality_drops if c in df_clean.columns], inplace=True, errors='ignore')

print(f"Cleanup complete. Remaining columns: {df_clean.shape[1]}")

Cleanup complete. Remaining columns: 89


### Phase 3 Checkpoint Export
We save the fully feature-engineered `df_clean` to disk. Future sessions can load this directly and skip straight to Phase 4 modeling without re-running the heavy Feature Engineering pipeline.

In [17]:
# --- Phase 3 Checkpoint Export ---
try:
    df_clean.to_parquet('wildlife_strikes_fe_engineered.parquet', engine='pyarrow')
    print(f"Phase 3 Checkpoint saved: 'wildlife_strikes_fe_engineered.parquet'  shape: {df_clean.shape}")
except Exception:
    df_clean.to_csv('wildlife_strikes_fe_engineered.csv', index=False)
    print(f"Phase 3 Checkpoint saved: 'wildlife_strikes_fe_engineered.csv'  shape: {df_clean.shape}")


Phase 3 Checkpoint saved: 'wildlife_strikes_fe_engineered.parquet'  shape: (307178, 89)
